# 02 - Relational schema analysis
Profiles each table, checks the keys, tests every foreign key and draws the schema.

In [ ]:
import sys, subprocess, importlib.util

# make sure the plotting packages are in this venv
for pkg in ["matplotlib", "networkx"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        importlib.invalidate_caches()

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# works whether the notebook is run from the project root or from notebooks/
here = Path.cwd().resolve()
PROJECT_ROOT = here.parent if here.name.lower() == "notebooks" else here
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "olist"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}. Run notebook 01 first.")

FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
dfs = {name: pd.read_csv(DATA_DIR / f) for name, f in FILES.items()}

for name, df in dfs.items():
    print(f"{name:22} rows={len(df):>9,}  cols={df.shape[1]}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. Table profile

In [ ]:
profile = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 1),
        "null_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    }
    for name, df in dfs.items()
])
display(profile)

# columns that actually have missing values
nulls = pd.concat(
    {name: df.isna().sum() for name, df in dfs.items()}, names=["table", "column"]
).rename("nulls").reset_index()
nulls = nulls[nulls["nulls"] > 0]
nulls["pct"] = [round(100 * n / len(dfs[t]), 2) for t, n in zip(nulls["table"], nulls["nulls"])]
display(nulls.sort_values("pct", ascending=False).reset_index(drop=True))

## 2. Primary keys
A key is valid if it has no nulls and no duplicates.

In [ ]:
CANDIDATE_KEYS = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],  # review_id alone repeats, see below
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
}

pk_rows = []
for table, keys in CANDIDATE_KEYS.items():
    df = dfs[table]
    null_rows = int(df[keys].isna().any(axis=1).sum())
    dup_rows = int(df.duplicated(subset=keys).sum())
    pk_rows.append({
        "table": table,
        "key": " + ".join(keys),
        "null_keys": null_rows,
        "duplicate_keys": dup_rows,
        "valid": null_rows == 0 and dup_rows == 0,
    })
pk_df = pd.DataFrame(pk_rows)
display(pk_df)

### Columns that look like keys but are not unique

In [ ]:
reviews, customers, payments, geo = dfs["reviews"], dfs["customers"], dfs["payments"], dfs["geolocation"]

print(f"reviews:   {reviews['review_id'].nunique():,} distinct review_id for {len(reviews):,} rows")
print(f"reviews:   {(reviews.groupby('order_id').size() > 1).sum()} orders have more than one review")
print(f"customers: {customers['customer_id'].nunique():,} customer_id vs {customers['customer_unique_id'].nunique():,} customer_unique_id")
print(f"payments:  {(payments.groupby('order_id').size() > 1).sum():,} orders have more than one payment record")
print(f"geolocation: {len(geo):,} rows, {geo['geolocation_zip_code_prefix'].nunique():,} zip prefixes, {geo.duplicated().sum():,} exact duplicate rows")

## 3. Foreign keys and cardinality
Orphans are child values with no parent. Parents without children are not an error, just worth knowing.

In [ ]:
RELATIONSHIPS = [
    # parent table, parent key, child table, child key
    ("customers", "customer_id", "orders", "customer_id"),
    ("orders", "order_id", "order_items", "order_id"),
    ("products", "product_id", "order_items", "product_id"),
    ("sellers", "seller_id", "order_items", "seller_id"),
    ("orders", "order_id", "payments", "order_id"),
    ("orders", "order_id", "reviews", "order_id"),
    ("category_translation", "product_category_name", "products", "product_category_name"),
]

fk_rows = []
for parent, pk, child, fk in RELATIONSHIPS:
    parent_vals = set(dfs[parent][pk].dropna())
    child_col = dfs[child][fk].dropna()
    per_parent = child_col.value_counts()
    fk_rows.append({
        "relationship": f"{parent}.{pk} -> {child}.{fk}",
        "orphan_rows": int((~child_col.isin(parent_vals)).sum()),
        "parents_without_children": len(parent_vals - set(child_col)),
        "max_children": int(per_parent.max()),
        "cardinality": "1:1" if per_parent.max() == 1 else "1:N",
    })
fk_df = pd.DataFrame(fk_rows)
display(fk_df)

## 4. Which orders show up in which tables?

In [ ]:
orders = dfs["orders"]
coverage = pd.DataFrame({
    "has_items": orders["order_id"].isin(dfs["order_items"]["order_id"]),
    "has_payment": orders["order_id"].isin(dfs["payments"]["order_id"]),
    "has_review": orders["order_id"].isin(dfs["reviews"]["order_id"]),
})
display(coverage.value_counts().rename("orders").reset_index())

# the orders without items are mostly unavailable / canceled ones
print("Orders without items, by status:")
display(orders.loc[~coverage["has_items"], "order_status"].value_counts().to_frame("orders"))

# which timestamps are empty for which status
time_cols = ["order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
status_nulls = orders.groupby("order_status")[time_cols].agg(lambda s: s.isna().sum())
status_nulls.insert(0, "orders", orders["order_status"].value_counts())
display(status_nulls.sort_values("orders", ascending=False))

## 5. Reference data: zip codes and categories

In [ ]:
geo_zips = set(geo["geolocation_zip_code_prefix"])
for table, col in [("customers", "customer_zip_code_prefix"), ("sellers", "seller_zip_code_prefix")]:
    no_geo = ~dfs[table][col].isin(geo_zips)
    print(f"{table}: {no_geo.sum()} rows have a zip prefix missing from geolocation")

# one zip prefix can map to many lat/lng pairs, so aggregate geolocation before joining it
coords = geo.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].nunique()
print(f"zip prefixes with more than one lat/lng: {(coords.max(axis=1) > 1).sum():,} of {len(coords):,}")

products, trans = dfs["products"], dfs["category_translation"]
no_category = products["product_category_name"].isna().sum()
untranslated = products["product_category_name"].dropna()
untranslated = untranslated[~untranslated.isin(trans["product_category_name"])]
print(f"products without a category: {no_category}")
print(f"products with an untranslated category: {len(untranslated)} ({sorted(untranslated.unique())})")

## 6. Schema diagram

In [ ]:
G = nx.DiGraph()
for parent, pk, child, fk in RELATIONSHIPS:
    G.add_edge(parent, child, label=pk)
G.add_edge("geolocation", "customers", label="zip prefix")
G.add_edge("geolocation", "sellers", label="zip prefix")

# hand-placed so the diagram reads left to right
pos = {
    "customers": (0, 2), "orders": (2, 2), "order_items": (4, 2),
    "payments": (2, 3.6), "reviews": (2, 0.4),
    "products": (6, 3), "sellers": (6, 1),
    "category_translation": (8, 3), "geolocation": (3, -1.2),
}

fig, ax = plt.subplots(figsize=(13, 7))
nx.draw_networkx_nodes(G, pos, node_size=4200, node_color="#cfe3f7", edgecolors="#2b5d8a", ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
nx.draw_networkx_edges(G, pos, arrowsize=18, node_size=4200, edge_color="#555", ax=ax)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "label"), font_size=8, ax=ax)
ax.set_title("Olist schema (arrow points from parent to child)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Save

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "schema_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for filename, frame in {"table_profile.csv": profile, "primary_key_validation.csv": pk_df,
                        "foreign_key_validation.csv": fk_df}.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)
print("Saved to", OUTPUT_DIR)